<a href="https://colab.research.google.com/github/it25101136/IT2011-Stress-Level-Prediction-/blob/Chenul/2_Dissanayake_DT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Shared Setup: Load Data and Split
from google.colab import drive
import pandas as pd
from sklearn.model_selection import train_test_split
import os

drive.mount('/content/drive', force_remount=True)

df_master = pd.read_csv('/content/drive/MyDrive/IT2011_G12_AI_Project/results/outputs/FINAL_ML_READY_DATA.csv')

# FIX: Prevent Data Leakage
# Initially, the model hit 1.0 (100%) accuracy because it was "cheating".
# 'stress_score' is the direct numerical equivalent of the target 'stress_level'.
# Keeping it allowed the AI to read the answer key instead of learning patterns.
# By dropping it, we force the AI to predict stress using actual student habits
# (sleep, screen time, etc.). The accuracy drops to a realistic 0.69, proving
# the model is genuinely working and not suffering from data leakage.

# FIX: Drop BOTH the target category and the direct score to prevent data leakage
X = df_master.drop(columns=['stress_level', 'stress_score'])
y = df_master['stress_level']

# 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Mounted at /content/drive
Training data shape: (2400, 7)
Testing data shape: (600, 7)


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

print("Executing Member 2 Model: Decision Tree")

param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

dt = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Optimum Parameters found: {grid_search.best_params_}")
print(f"Validation Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

pd.DataFrame({'Model': ['Decision Tree'], 'Accuracy': [accuracy]}).to_csv(
    '/content/drive/MyDrive/IT2011_G12_AI_Project/results/outputs/Score_Dissanayake.csv', index=False)

Executing Member 2 Model: Decision Tree
Optimum Parameters found: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 2}
Validation Accuracy: 0.6317

Classification Report:
               precision    recall  f1-score   support

           0       0.44      0.26      0.33        62
           1       0.68      0.63      0.66       239
           2       0.62      0.71      0.66       299

    accuracy                           0.63       600
   macro avg       0.58      0.53      0.55       600
weighted avg       0.63      0.63      0.62       600

